# QSS 45 Final Project — Models


### Models
1. Linear Regression for out-of-sample prediction
2. Gradient Boosting Regressor for out-of-sample prediction
3. Borough-only OLS with HC3 robust standard errors
4. Full OLS with socioeconomic controls and HC3 robust standard errors


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

# This allows the notebook to work from either the project root
# or from inside the code/ folder.
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "../data/qss45_bronx_manhattan_clean.csv").exists():
    DATA_DIR = (CURRENT_DIR / "../data").resolve()
    OUTPUT_DIR = (CURRENT_DIR / "../output").resolve()
elif (CURRENT_DIR / "data/qss45_bronx_manhattan_clean.csv").exists():
    DATA_DIR = (CURRENT_DIR / "data").resolve()
    OUTPUT_DIR = (CURRENT_DIR / "output").resolve()
else:
    raise FileNotFoundError(
        "Could not find qss45_bronx_manhattan_clean.csv. "
        "Run this notebook from the project root or code/ folder."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)


## 1. Load cleaned data

In [ ]:
data = pd.read_csv(DATA_DIR / "qss45_bronx_manhattan_clean.csv")

data["Bronx"] = (data["Borough"] == "Bronx").astype(int)

print("Rows:", len(data))
print(data["Borough"].value_counts())
data.head()


## 2. Define predictors and outcome

In [ ]:
features = [
    "Transit",
    "Income",
    "Poverty",
    "Unemployment",
    "WorkAtHome",
    "Bronx"
]

target = "MeanCommute"

X = data[features]
y = data[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 3. Create the same train/test split for both predictive models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=45,
    stratify=data["Borough"]
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 4. Predictive Model 1 — Linear Regression

Linear regression serves as the classical predictive benchmark. The predictors are standardized inside a pipeline because they use different measurement scales.


In [ ]:
linear_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_train_predictions = linear_model.predict(X_train)
linear_test_predictions = linear_model.predict(X_test)

linear_train_r2 = r2_score(y_train, linear_train_predictions)
linear_test_r2 = r2_score(y_test, linear_test_predictions)
linear_train_rmse = mean_squared_error(y_train, linear_train_predictions) ** 0.5
linear_test_rmse = mean_squared_error(y_test, linear_test_predictions) ** 0.5

print("Linear Regression")
print("-----------------")
print("Train R²:", round(linear_train_r2, 3))
print("Test R²:", round(linear_test_r2, 3))
print("Train RMSE:", round(linear_train_rmse, 3))
print("Test RMSE:", round(linear_test_rmse, 3), "minutes")


## 5. Predictive Model 2 — Gradient Boosting

Gradient boosting is a nonlinear ensemble method built from many small decision trees. Unlike linear regression, it can model nonlinear relationships and interactions without requiring them to be specified manually.


In [ ]:
gradient_boosting = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    subsample=0.90,
    random_state=45
)

gradient_boosting.fit(X_train, y_train)

gb_train_predictions = gradient_boosting.predict(X_train)
gb_test_predictions = gradient_boosting.predict(X_test)

gb_train_r2 = r2_score(y_train, gb_train_predictions)
gb_test_r2 = r2_score(y_test, gb_test_predictions)
gb_train_rmse = mean_squared_error(y_train, gb_train_predictions) ** 0.5
gb_test_rmse = mean_squared_error(y_test, gb_test_predictions) ** 0.5

print("Gradient Boosting")
print("-----------------")
print("Train R²:", round(gb_train_r2, 3))
print("Test R²:", round(gb_test_r2, 3))
print("Train RMSE:", round(gb_train_rmse, 3))
print("Test RMSE:", round(gb_test_rmse, 3), "minutes")


## 6. Compare out-of-sample predictive performance

In [ ]:
ml_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Gradient Boosting"],
    "Train_R2": [linear_train_r2, gb_train_r2],
    "Test_R2": [linear_test_r2, gb_test_r2],
    "Train_RMSE": [linear_train_rmse, gb_train_rmse],
    "Test_RMSE": [linear_test_rmse, gb_test_rmse]
}).round(3)

ml_comparison


In [ ]:
ml_comparison.to_csv(
    OUTPUT_DIR / "ml_model_comparison.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "ml_model_comparison.csv")


In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    ml_comparison["Model"],
    ml_comparison["Test_R2"]
)

plt.ylabel("Test R²")
plt.title("Out-of-Sample Predictive Performance")
plt.ylim(0, 1)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "final_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 7. Actual versus predicted commute time

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    y_test,
    linear_test_predictions,
    alpha=0.55,
    label="Linear Regression"
)

plt.scatter(
    y_test,
    gb_test_predictions,
    alpha=0.55,
    label="Gradient Boosting"
)

minimum = min(
    y_test.min(),
    linear_test_predictions.min(),
    gb_test_predictions.min()
)

maximum = max(
    y_test.max(),
    linear_test_predictions.max(),
    gb_test_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Mean Commute Time (Minutes)")
plt.ylabel("Predicted Mean Commute Time (Minutes)")
plt.title("Actual vs. Predicted Commute Time")
plt.legend()

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "actual_vs_predicted_models.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 8. Gradient Boosting feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "Variable": features,
    "Importance": gradient_boosting.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance.round(4)


In [ ]:
feature_importance.to_csv(
    OUTPUT_DIR / "gradient_boosting_feature_importance.csv",
    index=False
)

plot_importance = feature_importance.sort_values("Importance")

plt.figure(figsize=(8, 5))

plt.barh(
    plot_importance["Variable"],
    plot_importance["Importance"]
)

plt.xlabel("Feature Importance")
plt.title("Gradient Boosting Feature Importance")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "gradient_boosting_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 9. OLS Model 1 — Borough only

Manhattan is the reference category. The Bronx coefficient therefore represents the unadjusted Bronx–Manhattan difference in mean commute time.


In [ ]:
X_bronx_only = sm.add_constant(data[["Bronx"]])

model1 = sm.OLS(
    y,
    X_bronx_only
).fit(cov_type="HC3")

model1.summary()


## 10. OLS Model 2 — Full controls

This model estimates the Bronx–Manhattan commute gap after controlling for public transit use, income, poverty, unemployment, and working from home.


In [ ]:
X_full = sm.add_constant(data[[
    "Bronx",
    "Transit",
    "Income",
    "Poverty",
    "Unemployment",
    "WorkAtHome"
]])

model2 = sm.OLS(
    y,
    X_full
).fit(cov_type="HC3")

model2.summary()


## 11. Compare the two OLS specifications

In [ ]:
raw_bronx_gap = model1.params["Bronx"]
adjusted_bronx_gap = model2.params["Bronx"]

gap_reduction = (
    (raw_bronx_gap - adjusted_bronx_gap)
    / raw_bronx_gap
) * 100

model_comparison = pd.DataFrame({
    "Model": [
        "Model 1: Borough only",
        "Model 2: Full controls"
    ],
    "Bronx_Coefficient": [
        raw_bronx_gap,
        adjusted_bronx_gap
    ],
    "R_squared": [
        model1.rsquared,
        model2.rsquared
    ],
    "Adjusted_R_squared": [
        model1.rsquared_adj,
        model2.rsquared_adj
    ]
}).round(3)

model_comparison


In [ ]:
print("Raw Bronx commute gap:", round(raw_bronx_gap, 2), "minutes")
print("Adjusted Bronx commute gap:", round(adjusted_bronx_gap, 2), "minutes")
print("Percent reduction:", round(gap_reduction, 1), "%")

model_comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)


## 12. Full OLS coefficient table

In [ ]:
model2_confidence = model2.conf_int()

full_ols_results = pd.DataFrame({
    "Variable": model2.params.index,
    "Coefficient": model2.params.values,
    "Std_Error": model2.bse.values,
    "P_Value": model2.pvalues.values,
    "CI_Low": model2_confidence[0].values,
    "CI_High": model2_confidence[1].values
})

full_ols_results.round(4)


In [ ]:
full_ols_results.to_csv(
    OUTPUT_DIR / "full_ols_results.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "full_ols_results.csv")


## 13. OLS coefficient plot

In [ ]:
plot_data = (
    full_ols_results[
        full_ols_results["Variable"] != "const"
    ]
    .copy()
    .sort_values("Coefficient")
)

errors = np.vstack([
    plot_data["Coefficient"] - plot_data["CI_Low"],
    plot_data["CI_High"] - plot_data["Coefficient"]
])

plt.figure(figsize=(8, 5))

plt.errorbar(
    plot_data["Coefficient"],
    plot_data["Variable"],
    xerr=errors,
    fmt="o",
    capsize=4
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel("OLS Coefficient")
plt.ylabel("")
plt.title("Full OLS Model with 95% Confidence Intervals")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ols_coefficient_plot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 14. OLS residual diagnostic

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    model2.fittedvalues,
    model2.resid,
    alpha=0.6
)

plt.axhline(0, linestyle="--")

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("OLS Residual Diagnostic")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ols_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 15. Variance Inflation Factors

In [ ]:
X_vif = sm.add_constant(data[[
    "Bronx",
    "Transit",
    "Income",
    "Poverty",
    "Unemployment",
    "WorkAtHome"
]])

vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_table.round(2)


In [ ]:
vif_table.to_csv(
    OUTPUT_DIR / "vif_table.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "vif_table.csv")


## 16. Final model summary

In [ ]:
print("QSS 45 FINAL MODEL SUMMARY")
print("----------------------------------------")

print("Observations:", len(data))

print()
print("PREDICTIVE MODELS")
print("Linear Regression Test R²:", round(linear_test_r2, 3))
print("Linear Regression Test RMSE:", round(linear_test_rmse, 3), "minutes")
print("Gradient Boosting Test R²:", round(gb_test_r2, 3))
print("Gradient Boosting Test RMSE:", round(gb_test_rmse, 3), "minutes")

print()
print("OLS INFERENCE")
print("Raw Bronx commute gap:", round(raw_bronx_gap, 2), "minutes")
print("Adjusted Bronx commute gap:", round(adjusted_bronx_gap, 2), "minutes")
print("Model 1 R²:", round(model1.rsquared, 3))
print("Model 2 R²:", round(model2.rsquared, 3))
print("Percent reduction in Bronx coefficient:", round(gap_reduction, 1), "%")

print()
print("All tables and figures saved to:", OUTPUT_DIR)
